# T5-Small Baseline Training Setup

This notebook trains the reusable `t5-small` baseline model for the role-aware crisis tweet summarization project.

The current project state contains a GPT4o-generated teacher-summary dataset with 6001 non-`Other` labelled records, found in:

`data/generated/gpt4o_initial_summaries_v0203.jsonl`

The goal of this notebook is to:

1. Clone the latest project repository
2. Install the required training/evaluation dependencies.
3. Prepare train/validation/test splits from the 6001 example summary dataset.
4. Train `t5-small` as the reusable baseline.
5. Save the trained checkpoint and metrics artifacts for later download/commit.

This first setup cell checks the Kaggle runtime, GPU availability, Python version, and working directory before we clone or install anything.

In [1]:
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", os.getcwd())
print("Kaggle working exists:", Path("/kaggle/working").exists())
print("Kaggle input exists:", Path("/kaggle/input").exists())

print("\nGPU check:")
!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Current working directory: /kaggle/working
Kaggle working exists: True
Kaggle input exists: True

GPU check:
Sun Jun 28 16:27:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8

## Clone The Project Repository

After cloning, we print the latest commit and inspect the key generated dataset that will be used for the baseline.

In [2]:
from pathlib import Path
import os

REPO_URL = "https://github.com/ffaustin17/role-aware-crisis-summarization.git"
REPO_DIR = Path("/kaggle/working/role-aware-crisis-summarization")

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Current directory: ", Path.cwd())

print("\nLatest commit: ")
!git log --oneline -5

print("\nRepository status: ")
!git status --short --branch

print("\nGenerated data files: ")
!ls -lh data/generated



Cloning into '/kaggle/working/role-aware-crisis-summarization'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 267 (delta 129), reused 206 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (267/267), 9.08 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Current directory:  /kaggle/working/role-aware-crisis-summarization

Latest commit: 
4e058cf (HEAD -> main, origin/main, origin/HEAD) Update project timeline with recent progress
0e24489 Add summary reward analysis reports
92e4836 Add initial successful GPT summaries dataset
3493606 Add 800 generated summaries
709dd9f Add 1200 generated summaries

Repository status: 
## main...origin/main

Generated data files: 
total 29M
-rw-r--r-- 1 root root 11K Jun 28 16:27 base_summaries_v1_test.jsonl
-rw-r--r-- 1 root root 15M Jun 28 16:27 gpt4o_initial_summaries_v0203.jsonl
-rw-r--r-- 1 root root 53K Jun 28 16:27 su

In [3]:
# Check definitive 6,001 summary dataset

import json
from pathlib import Path

path = Path("data/generated/gpt4o_initial_summaries_v0203.jsonl")
count = 0
statuses = {}
roles = {}
for line in path.open(encoding="utf-8"):
    if not line.strip():
        continue
    rec = json.loads(line)
    count += 1
    statuses[rec.get("generation_status")] = statuses.get(rec.get("generation_status"), 0) + 1
    roles[rec.get("role")] = roles.get(rec.get("role"), 0) + 1

print("exists:", path.exists())
print("records:", count)
print("statuses:", statuses)
print("role labels:", dict(sorted(roles.items())))

exists: True
records: 6001
statuses: {'success': 6001}
role labels: {'EMS': 593, 'Firefighter': 1023, 'Firefighter/EMS': 153, 'Police': 3280, 'Police/EMS': 401, 'Police/Firefighter': 340, 'Police/Firefighter/EMS': 211}


## Install Training And Evaluation Dependencies

This notebook needs the Hugging Face training stack plus evaluation metrics.

Kaggle already includes many ML libraries, so we install only the packages required by the project scripts instead of recreating the full local `uv` environment.

The expected dependency groups are:

- `transformers` and `accelerate` for T5 training
- `datasets` for dataset handling
- `evaluate`, `rouge_score`, and `sacrebleu` for summarization metrics
- `bert_score` for semantic similarity evaluation
- `sentencepiece` for T5 tokenization support

Some Kaggle CUDA-related dependency warnings can appear during installation. We only care that the imports in the next cell succeed.

In [4]:
!pip install -q \
  "transformers>=4.40" \
  "accelerate>=0.30" \
  "datasets>=2.19" \
  "evaluate>=0.4" \
  "rouge_score>=0.1.2" \
  "sacrebleu>=2.4" \
  "bert_score>=0.3.13" \
  "sentencepiece>=0.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cud

## Verify Dependency Imports

This cell confirms that the installed packages can be imported in the current Kaggle runtime.

If these imports pass, the notebook is ready to prepare the updated 6,001-example T5 baseline dataset.

In [5]:
import torch
import transformers
import datasets
import evaluate
import sentencepiece
import accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("accelerate:", accelerate.__version__)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

torch: 2.10.0+cu128
cuda available: True
gpu count: 2
transformers: 5.0.0
datasets: 4.8.5
evaluate: 0.4.6
accelerate: 1.13.0
GPU 0: Tesla T4
GPU 1: Tesla T4


## Prepare The Updated T5 Baseline Dataset

The first T5 baseline was trained on 4,001 GPT-generated summaries. The project now has a definitive successful-only GPT teacher dataset with 6,001 non-`Other` examples:

`data/generated/gpt4o_initial_summaries_v0203.jsonl`

This cell creates a new fixed `80/10/10` train/validation/test split for the updated baseline.

Important choices:

- The output directory is `data/modeling/t5_baseline_v2/`.
- The split is stratified by the exact raw `role` label.
- Multi-role labels such as `Police/EMS` remain single categories.
- The input field is `input_text`.
- The target field is `final_base_summary_text`, written as `target_text` in the modeling split.

In [6]:
from pathlib import Path

DATASET_V2_INPUT = Path("data/generated/gpt4o_initial_summaries_v0203.jsonl")
DATASET_V2_DIR = Path("data/modeling/t5_baseline_v2")

print("Input exists:", DATASET_V2_INPUT.exists(), DATASET_V2_INPUT)
print("Output dir:", DATASET_V2_DIR)

!python scripts/prepare_t5_baseline_data.py \
  --input data/generated/gpt4o_initial_summaries_v0203.jsonl \
  --output-dir data/modeling/t5_baseline_v2 \
  --seed 42

Input exists: True data/generated/gpt4o_initial_summaries_v0203.jsonl
Output dir: data/modeling/t5_baseline_v2
Clean successful examples: 6001
train: 4800
validation: 600
test: 601
Wrote baseline splits to: data/modeling/t5_baseline_v2


## Inspect The New Split

This cell verifies that the split was created correctly.

Expected approximate counts from 6,001 examples:

- train: about 4,800
- validation: about 600
- test: about 601

The split metadata and role distribution help confirm that the split is reproducible and stratified by exact role label.

In [7]:
import json
from pathlib import Path

split_dir = Path("data/modeling/t5_baseline_v2")

print("Files:")
!ls -lh data/modeling/t5_baseline_v2

print("\nSplit metadata:")
metadata = json.loads((split_dir / "split_metadata.json").read_text())
print(json.dumps(metadata, indent=2)[:3000])

print("\nRole distribution:")
!cat data/modeling/t5_baseline_v2/role_distribution.csv

print("\nLine counts:")
!wc -l data/modeling/t5_baseline_v2/train.jsonl \
      data/modeling/t5_baseline_v2/validation.jsonl \
      data/modeling/t5_baseline_v2/test.jsonl

print("\nFirst training record:")
!head -n 1 data/modeling/t5_baseline_v2/train.jsonl

Files:
total 4.9M
-rw-r--r-- 1 root root  544 Jun 28 16:27 role_distribution.csv
-rw-r--r-- 1 root root 1010 Jun 28 16:27 split_metadata.json
-rw-r--r-- 1 root root 492K Jun 28 16:27 test.jsonl
-rw-r--r-- 1 root root 3.9M Jun 28 16:27 train.jsonl
-rw-r--r-- 1 root root 492K Jun 28 16:27 validation.jsonl

Split metadata:
{
  "input_path": "data/generated/gpt4o_initial_summaries_v0203.jsonl",
  "output_dir": "data/modeling/t5_baseline_v2",
  "random_seed": 42,
  "split_strategy": "80/10/10 stratified by exact role",
  "input_field": "input_text",
  "target_field": "final_base_summary_text",
  "record_counts": {
    "train": 4800,
    "validation": 600,
    "test": 601
  },
  "role_counts": {
    "train": {
      "EMS": 474,
      "Firefighter": 818,
      "Firefighter/EMS": 122,
      "Police": 2624,
      "Police/EMS": 321,
      "Police/Firefighter": 272,
      "Police/Firefighter/EMS": 169
    },
    "validation": {
      "EMS": 59,
      "Firefighter": 102,
      "Firefighter/EMS": 1

## Training Smoke Test

Before launching full training, this cell runs a tiny T5-small training job on a very small subset of the new split.

This is not a meaningful model. It is a wiring test for:

- loading `data/modeling/t5_baseline_v2`
- downloading/loading `t5-small`
- tokenizing `input_text` and `target_text`
- running Hugging Face `Seq2SeqTrainer`
- writing a checkpoint directory

If this passes, the notebook is ready for the full baseline v2 training run.

In [8]:
SMOKE_MODEL_DIR = "/kaggle/working/t5_small_baseline_v2_smoke"

!rm -rf {SMOKE_MODEL_DIR}

!python scripts/train_t5_baseline.py \
  --data-dir data/modeling/t5_baseline_v2 \
  --output-dir {SMOKE_MODEL_DIR} \
  --model-name t5-small \
  --num-train-epochs 1 \
  --train-batch-size 4 \
  --eval-batch-size 4 \
  --gradient-accumulation-steps 2 \
  --max-train-samples 32 \
  --max-eval-samples 16 \
  --logging-steps 5 \
  --save-total-limit 1

Generating train split: 4800 examples [00:00, 125466.83 examples/s]
Generating validation split: 600 examples [00:00, 104418.17 examples/s]
config.json: 1.21kB [00:00, 4.12MB/s]
tokenizer_config.json: 2.32kB [00:00, 6.91MB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 24.0MB/s]
tokenizer.json: 1.39MB [00:00, 22.0MB/s]
model.safetensors: 100%|██████████████████████| 242M/242M [00:01<00:00, 169MB/s]
Loading weights: 100%|█| 131/131 [00:00<00:00, 1482.52it/s, Materializing param=
  0%|                                                     | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
100%|█████████████████████████████████████████████| 2/2 [00:02<00:00,  1.01s/it]

{'eval_loss': '9.577', 'eval_runtime': '0.1412', 'eval_samples

## Inspect Smoke-Test Checkpoint

This cell confirms that the smoke-test checkpoint directory was created.

If model/tokenizer files exist, we can proceed to full training.

In [9]:
from pathlib import Path

smoke_dir = Path("/kaggle/working/t5_small_baseline_v2_smoke")
print("Smoke dir exists:", smoke_dir.exists())

if smoke_dir.exists():
    print("Smoke checkpoint files:")
    !find /kaggle/working/t5_small_baseline_v2_smoke -maxdepth 2 -type f | sort | sed 's#^#/##' | head -50

Smoke dir exists: True
Smoke checkpoint files:
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/config.json
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/generation_config.json
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/model.safetensors
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/optimizer.pt
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/rng_state.pth
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/scaler.pt
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/scheduler.pt
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/tokenizer_config.json
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/tokenizer.json
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/trainer_state.json
//kaggle/working/t5_small_baseline_v2_smoke/checkpoint-2/training_args.bin
//kaggle/working/t5_small_baseline_v2_smoke/config.json
//kaggle/working/t5_small_baseline_v2_smoke/generation_config.json
//kaggle/working/t5_small_

## Full T5-Small Baseline v2 Training

This cell trains the updated reusable `t5-small` baseline on the 6,001-example GPT teacher dataset split.

This is the real baseline v2 checkpoint.

Training setup:

- Dataset: `data/modeling/t5_baseline_v2`
- Train examples: 4,800
- Validation examples: 600
- Model: `t5-small`
- Epochs: 3
- Input: runtime `input_text`
- Target: GPT teacher `target_text`
- Output checkpoint: `/kaggle/working/t5_small_baseline_v2`

The resulting checkpoint is a frozen reusable model artifact. It should not be committed directly to Git because it is large, but it should be saved/downloaded.

In [10]:
FULL_MODEL_DIR = "/kaggle/working/t5_small_baseline_v2"

!rm -rf {FULL_MODEL_DIR}

!python scripts/train_t5_baseline.py \
  --data-dir data/modeling/t5_baseline_v2 \
  --output-dir {FULL_MODEL_DIR} \
  --model-name t5-small \
  --num-train-epochs 3 \
  --train-batch-size 8 \
  --eval-batch-size 8 \
  --gradient-accumulation-steps 1 \
  --learning-rate 5e-5 \
  --weight-decay 0.01 \
  --logging-steps 25 \
  --save-total-limit 2

Loading weights: 100%|█| 131/131 [00:00<00:00, 1655.02it/s, Materializing param=
  0%|                                                   | 0/900 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
{'loss': '8.38', 'grad_norm': '7.261', 'learning_rate': '4.867e-05', 'epoch': '0.08333'}
{'loss': '6.707', 'grad_norm': '4.66', 'learning_rate': '4.728e-05', 'epoch': '0.1667'}
{'loss': '5.822', 'grad_norm': '4.629', 'learning_rate': '4.589e-05', 'epoch': '0.25'}
{'loss': '5.403', 'grad_norm': '6.729', 'learning_rate': '4.45e-05', 'epoch': '0.3333'}
{'loss': '5.085', 'grad_norm': '6.09', 'learning_rate': '4.311e-05', 'epoch': '0.4167'}
{'loss': '4.889', 'grad_norm': '9.975', 'learning_rate': '4.172e-05', 'epoch': '0.5'}
{'loss': '4.666', 'grad_norm': '8.164', 'learni

## Inspect Full Baseline v2 Checkpoint

This cell confirms that the full baseline v2 checkpoint was saved successfully.

The top-level checkpoint directory should include model weights, tokenizer files, config files, and training arguments.

In [11]:
from pathlib import Path

full_model_dir = Path("/kaggle/working/t5_small_baseline_v2")
print("Full model dir exists:", full_model_dir.exists())

if full_model_dir.exists():
    print("\nTop-level files:")
    !find /kaggle/working/t5_small_baseline_v2 -maxdepth 1 -type f | sort

    print("\nCheckpoint subdirectories:")
    !find /kaggle/working/t5_small_baseline_v2 -maxdepth 1 -type d | sort

    print("\nDisk usage:")
    !du -sh /kaggle/working/t5_small_baseline_v2

Full model dir exists: True

Top-level files:
/kaggle/working/t5_small_baseline_v2/config.json
/kaggle/working/t5_small_baseline_v2/generation_config.json
/kaggle/working/t5_small_baseline_v2/model.safetensors
/kaggle/working/t5_small_baseline_v2/tokenizer_config.json
/kaggle/working/t5_small_baseline_v2/tokenizer.json
/kaggle/working/t5_small_baseline_v2/training_args.bin

Checkpoint subdirectories:
/kaggle/working/t5_small_baseline_v2
/kaggle/working/t5_small_baseline_v2/checkpoint-600
/kaggle/working/t5_small_baseline_v2/checkpoint-900

Disk usage:
1.6G	/kaggle/working/t5_small_baseline_v2


## Evaluate Baseline v2 On The Held-Out Test Split

This cell evaluates the full `t5-small` baseline v2 checkpoint on the new 601-example held-out test split.

The evaluation script generates predictions and computes:

- ROUGE-1 F1
- ROUGE-2 F1
- ROUGE-L F1
- SacreBLEU
- BERTScore precision/recall/F1

The generated predictions are saved as JSONL so they can later be scored with the role-aware reward function.

In [12]:
!python scripts/evaluate_t5_baseline.py \
  --data-dir data/modeling/t5_baseline_v2 \
  --model-dir /kaggle/working/t5_small_baseline_v2 \
  --metrics-csv reports/tables/t5_small_baseline_v2_metrics.csv \
  --metrics-md reports/tables/t5_small_baseline_v2_metrics.md \
  --predictions data/modeling/t5_baseline_v2/test_predictions.jsonl \
  --batch-size 8 \
  --num-beams 4

Loading weights: 100%|█| 131/131 [00:00<00:00, 1597.77it/s, Materializing param=
config.json: 100%|█████████████████████████████| 482/482 [00:00<00:00, 1.85MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 129kB/s]
vocab.json: 899kB [00:00, 61.6MB/s]
merges.txt: 456kB [00:00, 40.2MB/s]
tokenizer.json: 1.36MB [00:00, 27.5MB/s]
model.safetensors: 100%|████████████████████| 1.42G/1.42G [00:10<00:00, 134MB/s]
Loading weights: 100%|█| 389/389 [00:00<00:00, 1474.56it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
poole

## Inspect Baseline v2 Evaluation Outputs

This cell displays the baseline v2 metrics table and confirms the prediction file has one row per held-out test example.

In [13]:
from pathlib import Path

metrics_md = Path("reports/tables/t5_small_baseline_v2_metrics.md")
metrics_csv = Path("reports/tables/t5_small_baseline_v2_metrics.csv")
predictions_path = Path("data/modeling/t5_baseline_v2/test_predictions.jsonl")

print("Metrics MD exists:", metrics_md.exists())
print("Metrics CSV exists:", metrics_csv.exists())
print("Predictions exist:", predictions_path.exists())

print("\nMetrics:")
print(metrics_md.read_text())

print("\nPrediction count:")
!wc -l data/modeling/t5_baseline_v2/test_predictions.jsonl

print("\nFirst prediction:")
!head -n 1 data/modeling/t5_baseline_v2/test_predictions.jsonl

Metrics MD exists: True
Metrics CSV exists: True
Predictions exist: True

Metrics:
| metric | value |
|---|---:|
| rouge1_f1 | 0.511072 |
| rouge2_f1 | 0.271986 |
| rougeL_f1 | 0.452822 |
| bleu | 19.952732 |
| bertscore_precision | 0.927297 |
| bertscore_recall | 0.923540 |
| bertscore_f1 | 0.925358 |
| num_examples | 601.000000 |


Prediction count:
601 data/modeling/t5_baseline_v2/test_predictions.jsonl

First prediction:
{"tweet_id": 3130, "source_row_id": 22897, "role": "Police", "disaster_type": "Collapse", "information_type": "Other Useful Information", "input_text": "Responder Roles: Police\nDisaster Type: Collapse\nTweet: Arrests over Dhaka building collapse: Two owners of garment factories in the building that collapsed in Bangladesh are ...  @tobeymonster\nSecondary Annotations: CPU\nInformation Type: Other Useful Information", "target_text": "Investigate possible criminal activity linked to the building collapse while maintaining scene security in Dhaka.", "prediction_text"

## Generate Baseline v2 Predictions For All 6,001 Examples

The held-out test predictions are useful for standard evaluation metrics, but the next project stage needs model-generated summaries for every input in the 6,001-example dataset.

This cell uses the trained `t5_small_baseline_v2` checkpoint to generate predictions for all examples across:

- train
- validation
- test

The output file is:

`data/modeling/t5_baseline_v2/all_predictions.jsonl`

These predictions will later be scored with the same role-aware reward function used for GPT teacher summaries. That enables direct paired comparison between:

- GPT teacher summary
- T5 baseline v2 summary

In [14]:
import json
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DATA_DIR = Path("data/modeling/t5_baseline_v2")
MODEL_DIR = Path("/kaggle/working/t5_small_baseline_v2")
OUTPUT_PATH = DATA_DIR / "all_predictions.jsonl"

def read_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

records = []
for split_name in ["train", "validation", "test"]:
    split_records = read_jsonl(DATA_DIR / f"{split_name}.jsonl")
    for record in split_records:
        record["split"] = split_name
    records.extend(split_records)

records = sorted(records, key=lambda r: int(r["tweet_id"]))

print("Total records:", len(records))
print("First tweet_id:", records[0]["tweet_id"])
print("Last tweet_id:", records[-1]["tweet_id"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device)
model.eval()

batch_size = 16
max_input_length = 512
max_generation_length = 128
num_beams = 4

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8") as out:
    for start in tqdm(range(0, len(records), batch_size), desc="Generating all predictions"):
        batch = records[start:start + batch_size]
        inputs = [record["input_text"] for record in batch]

        encoded = tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **encoded,
                max_length=max_generation_length,
                num_beams=num_beams,
            )

        decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for record, prediction in zip(batch, decoded):
            output_record = {
                "tweet_id": record["tweet_id"],
                "source_row_id": record.get("source_row_id"),
                "split": record["split"],
                "role": record.get("role"),
                "disaster_type": record.get("disaster_type"),
                "information_type": record.get("information_type"),
                "input_text": record["input_text"],
                "target_text": record["target_text"],
                "prediction_text": prediction.strip(),
            }
            out.write(json.dumps(output_record, ensure_ascii=False) + "\n")

print("Wrote:", OUTPUT_PATH)

Total records: 6001
First tweet_id: 1
Last tweet_id: 6001
Device: cuda


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating all predictions:   0%|          | 0/376 [00:00<?, ?it/s]

Wrote: data/modeling/t5_baseline_v2/all_predictions.jsonl


## Inspect Full Baseline v2 Predictions

This cell verifies that the full prediction file has exactly 6,001 rows and includes train/validation/test split labels.

These predictions are the main artifact needed before reward scoring the baseline v2 model at scale.

In [15]:
from collections import Counter
import json
from pathlib import Path

path = Path("data/modeling/t5_baseline_v2/all_predictions.jsonl")

count = 0
splits = Counter()
roles = Counter()
empty_predictions = 0

with path.open(encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        count += 1
        splits[rec.get("split")] += 1
        roles[rec.get("role")] += 1
        if not rec.get("prediction_text", "").strip():
            empty_predictions += 1

print("Prediction rows:", count)
print("Split counts:", dict(splits))
print("Role counts:", dict(sorted(roles.items())))
print("Empty predictions:", empty_predictions)

print("\nFile size:")
!ls -lh data/modeling/t5_baseline_v2/all_predictions.jsonl

print("\nFirst prediction:")
!head -n 1 data/modeling/t5_baseline_v2/all_predictions.jsonl

print("\nLast prediction:")
!tail -n 1 data/modeling/t5_baseline_v2/all_predictions.jsonl

Prediction rows: 6001
Split counts: {'test': 601, 'train': 4800, 'validation': 600}
Role counts: {'EMS': 593, 'Firefighter': 1023, 'Firefighter/EMS': 153, 'Police': 3280, 'Police/EMS': 401, 'Police/Firefighter': 340, 'Police/Firefighter/EMS': 211}
Empty predictions: 0

File size:
-rw-r--r-- 1 root root 4.0M Jun 28 16:34 data/modeling/t5_baseline_v2/all_predictions.jsonl

First prediction:
{"tweet_id": 1, "source_row_id": 495, "split": "test", "role": "Firefighter", "disaster_type": "Wildfire", "information_type": "Affected individuals", "input_text": "Responder Roles: Firefighter\nDisaster Type: Wildfire\nTweet: Yikes, that smoke is too close for comfort. Stay safe, #Boulder! #flagstafffire\nSecondary Annotations: FC\nInformation Type: Affected individuals", "target_text": "Firefighters should assess smoke proximity and potential fire spread near Boulder, preparing for possible containment efforts.", "prediction_text": "Firefighters should assess fire hazards and ensure safety in areas

## Package Git-Friendly Baseline v2 Artifacts

This cell packages the baseline v2 artifacts that should be brought back into the Git repository.

Included:

- train/validation/test split files
- full 6,001 prediction file
- held-out test prediction file
- split metadata
- role distribution
- baseline v2 metrics CSV/Markdown

The trained model checkpoint is not included here because it is large and should be preserved separately as an external model artifact.

In [16]:
RESULTS_ZIP = "/kaggle/working/t5_baseline_v2_results.zip"

!rm -f {RESULTS_ZIP}

!zip -r {RESULTS_ZIP} \
  data/modeling/t5_baseline_v2 \
  reports/tables/t5_small_baseline_v2_metrics.csv \
  reports/tables/t5_small_baseline_v2_metrics.md

print("\nPackaged Git-friendly results:")
!ls -lh {RESULTS_ZIP}

print("\nContents preview:")
!unzip -l {RESULTS_ZIP} | head -80

  adding: data/modeling/t5_baseline_v2/ (stored 0%)
  adding: data/modeling/t5_baseline_v2/train.jsonl (deflated 86%)
  adding: data/modeling/t5_baseline_v2/split_metadata.json (deflated 66%)
  adding: data/modeling/t5_baseline_v2/all_predictions.jsonl (deflated 84%)
  adding: data/modeling/t5_baseline_v2/validation.jsonl (deflated 86%)
  adding: data/modeling/t5_baseline_v2/test.jsonl (deflated 86%)
  adding: data/modeling/t5_baseline_v2/role_distribution.csv (deflated 69%)
  adding: data/modeling/t5_baseline_v2/test_predictions.jsonl (deflated 83%)
  adding: reports/tables/t5_small_baseline_v2_metrics.csv (deflated 35%)
  adding: reports/tables/t5_small_baseline_v2_metrics.md (deflated 41%)

Packaged Git-friendly results:
-rw-r--r-- 1 root root 1.4M Jun 28 16:34 /kaggle/working/t5_baseline_v2_results.zip

Contents preview:
Archive:  /kaggle/working/t5_baseline_v2_results.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-06-28 16:31   data/model

## Package The Baseline v2 Model Checkpoint

This cell packages the trained `t5-small` baseline v2 checkpoint separately.

This checkpoint is the frozen reusable model artifact produced by the notebook. It is intentionally separate from the Git-friendly result package because it is much larger than the CSV/JSONL/report artifacts.

In [17]:
MODEL_ZIP = "/kaggle/working/t5_small_baseline_v2_checkpoint.zip"

!rm -f {MODEL_ZIP}

!zip -r {MODEL_ZIP} /kaggle/working/t5_small_baseline_v2

print("\nPackaged model checkpoint:")
!ls -lh {MODEL_ZIP}

print("\nContents preview:")
!unzip -l {MODEL_ZIP} | head -80

  adding: kaggle/working/t5_small_baseline_v2/ (stored 0%)
  adding: kaggle/working/t5_small_baseline_v2/tokenizer_config.json (deflated 83%)
  adding: kaggle/working/t5_small_baseline_v2/config.json (deflated 63%)
  adding: kaggle/working/t5_small_baseline_v2/generation_config.json (deflated 28%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/ (stored 0%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/scheduler.pt (deflated 61%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/tokenizer_config.json (deflated 83%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/config.json (deflated 63%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/trainer_state.json (deflated 74%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/generation_config.json (deflated 28%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/training_args.bin (deflated 53%)
  adding: kaggle/working/t5_small_baseline_v2/checkpoint-600/tok

## Optional Base64 Transfer Files

If the Kaggle interface makes it difficult to download zip files directly, this cell creates `.b64` versions of the two packages.

These text files can be copied out of Kaggle and decoded locally with PowerShell.

In [18]:
!base64 /kaggle/working/t5_baseline_v2_results.zip > /kaggle/working/t5_baseline_v2_results.b64
!base64 /kaggle/working/t5_small_baseline_v2_checkpoint.zip > /kaggle/working/t5_small_baseline_v2_checkpoint.b64

print("Base64 files:")
!ls -lh /kaggle/working/*.b64

Base64 files:
-rw-r--r-- 1 root root 1.9M Jun 28 16:35 /kaggle/working/t5_baseline_v2_results.b64
-rw-r--r-- 1 root root 2.0G Jun 28 16:35 /kaggle/working/t5_small_baseline_v2_checkpoint.b64
